## Hybrid Tensor Persistence: Cloudflare + S3

This example uses both cloud pools in one workflow:
- A **large** torch tensor is uploaded to **Cloudflare R2**
- Multiple **small** torch tensors are uploaded to **AWS S3**

### Credentials
Reads credentials from:
`laila.args`


In [ ]:
import laila
import torch


In [ ]:
from laila.pool import S3Pool, CloudflarePool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own cloud credentials before running.
laila.args.AWS_BUCKET_NAME = "your-s3-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-aws-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-aws-secret-access-key"
laila.args.AWS_REGION = "us-east-1"
laila.args.CLOUDFLARE_ACCOUNT_ID = "your-account-id"
laila.args.CLOUDFLARE_ACCESS_KEY_ID = "your-cloudflare-access-key-id"
laila.args.CLOUDFLARE_SECRET_ACCESS_KEY = "your-cloudflare-secret-access-key"
laila.args.CLOUDFLARE_BUCKET_NAME = "your-r2-bucket-name"

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
)

cloudflare_pool = CloudflarePool(
    account_id=laila.args.CLOUDFLARE_ACCOUNT_ID,
    access_key_id=laila.args.CLOUDFLARE_ACCESS_KEY_ID,
    secret_access_key=laila.args.CLOUDFLARE_SECRET_ACCESS_KEY,
    bucket_name=laila.args.CLOUDFLARE_BUCKET_NAME,
)


In [ ]:
laila.memory.extend(s3_pool, pool_nickname="hybrid_s3_pool")
laila.memory.extend(cloudflare_pool, pool_nickname="hybrid_cloudflare_pool")


In [ ]:
# Large tensor for Cloudflare (R2)
# ~67 MB in float32 (4096 x 4096)
large_tensor = torch.randn(4096, 4096, dtype=torch.float32)
large_entry = laila.constant(data=large_tensor)

print("Large tensor shape:", tuple(large_tensor.shape))
print("Large tensor global_id:", large_entry.global_id)


In [ ]:
# Multiple small tensors for S3
small_entries = [
    laila.constant(data=torch.randn(64, 64, dtype=torch.float32))
    for _ in range(12)
]

print("Small tensor entries:", len(small_entries))
print("First small tensor global_id:", small_entries[0].global_id)


In [ ]:
with laila.guarantee:
    laila.memorize(large_entry, pool_nickname="hybrid_cloudflare_pool")

print("Large tensor uploaded to Cloudflare.")


In [ ]:
with laila.guarantee:
    laila.memorize(small_entries, pool_nickname="hybrid_s3_pool")

print("Small tensors uploaded to S3:", len(small_entries))


In [ ]:
sample_small_entry = small_entries[0]
with laila.guarantee:
    recovered_large = laila.remember(large_entry.global_id, pool_nickname="hybrid_cloudflare_pool")
    recovered_small = laila.remember(sample_small_entry.global_id, pool_nickname="hybrid_s3_pool")

assert torch.equal(recovered_large.data, large_tensor)
assert torch.equal(recovered_small.data, sample_small_entry.data)

print("Recovery checks passed for Cloudflare and S3.")


In [ ]:
with laila.guarantee:
    laila.forget(large_entry.global_id, pool_nickname="hybrid_cloudflare_pool")
    laila.forget([e.global_id for e in small_entries], pool_nickname="hybrid_s3_pool")

print("Cleanup complete.")
